In [1]:
import pandas as pd
import numpy as np

RAW_DATA_PATH = "../data/raw/telco_customer_churn.csv"
PROCESSED_DATA_PATH = "../data/processed/telco_customer_churn_clean.csv"

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Basic Cleaning & Type Fixes

In [4]:
df = df.drop(columns=["customerID"])

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

((df["TotalCharges"].isna()) & (df["tenure"] == 0)).sum()

np.int64(11)

Handle Missing TotalCharges Due to Tenure = 0

In [5]:
df = df.dropna(subset=["TotalCharges"])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   object 
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

Missing Value Assertion

In [6]:
assert df.isna().sum().sum() == 0

Define Target & Feature Groups

In [7]:
TARGET = "Churn"

categorical_features = [
    col for col in df.select_dtypes(include="object").columns
    if col != TARGET
]

numeric_features = df.select_dtypes(exclude="object").columns.tolist()

print("Target:", TARGET)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Target: Churn
Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


Categorical Value Distributions

In [8]:
for col in categorical_features:
    print(f"\n{col}")
    print("Counts:", df[col].value_counts().to_dict())


gender
Counts: {'Male': 3549, 'Female': 3483}

SeniorCitizen
Counts: {'No': 5890, 'Yes': 1142}

Partner
Counts: {'No': 3639, 'Yes': 3393}

Dependents
Counts: {'No': 4933, 'Yes': 2099}

PhoneService
Counts: {'Yes': 6352, 'No': 680}

MultipleLines
Counts: {'No': 3385, 'Yes': 2967, 'No phone service': 680}

InternetService
Counts: {'Fiber optic': 3096, 'DSL': 2416, 'No': 1520}

OnlineSecurity
Counts: {'No': 3497, 'Yes': 2015, 'No internet service': 1520}

OnlineBackup
Counts: {'No': 3087, 'Yes': 2425, 'No internet service': 1520}

DeviceProtection
Counts: {'No': 3094, 'Yes': 2418, 'No internet service': 1520}

TechSupport
Counts: {'No': 3472, 'Yes': 2040, 'No internet service': 1520}

StreamingTV
Counts: {'No': 2809, 'Yes': 2703, 'No internet service': 1520}

StreamingMovies
Counts: {'No': 2781, 'Yes': 2731, 'No internet service': 1520}

Contract
Counts: {'Month-to-month': 3875, 'Two year': 1685, 'One year': 1472}

PaperlessBilling
Counts: {'Yes': 4168, 'No': 2864}

PaymentMethod
Counts:

PhoneService → MultipleLines Dependency Check

In [9]:
# Crosstab to inspect relationship
ct_phone = pd.crosstab(df["PhoneService"], df["MultipleLines"])
print(ct_phone)

# Check violations
violations_phone = df[
    (df["PhoneService"] == "No") &
    (df["MultipleLines"] != "No phone service")
]

print(f"\nNumber of dependency violations (PhoneService → MultipleLines): {len(violations_phone)}")

MultipleLines    No  No phone service   Yes
PhoneService                               
No                0               680     0
Yes            3385                 0  2967

Number of dependency violations (PhoneService → MultipleLines): 0


InternetService Dependency Checks

In [10]:
internet_dependent_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in internet_dependent_cols:
    print(f"\nChecking dependency: InternetService → {col}")
    
    ct = pd.crosstab(df["InternetService"], df[col])
    print(ct)
    
    violations = df[
        (df["InternetService"] == "No") &
        (df[col] != "No internet service")
    ]
    
    print(f"Violations: {len(violations)}")


Checking dependency: InternetService → OnlineSecurity
OnlineSecurity     No  No internet service   Yes
InternetService                                 
DSL              1240                    0  1176
Fiber optic      2257                    0   839
No                  0                 1520     0
Violations: 0

Checking dependency: InternetService → OnlineBackup
OnlineBackup       No  No internet service   Yes
InternetService                                 
DSL              1334                    0  1082
Fiber optic      1753                    0  1343
No                  0                 1520     0
Violations: 0

Checking dependency: InternetService → DeviceProtection
DeviceProtection    No  No internet service   Yes
InternetService                                  
DSL               1355                    0  1061
Fiber optic       1739                    0  1357
No                   0                 1520     0
Violations: 0

Checking dependency: InternetService → TechSupport
T

Hierarchical Category Flattening

In [11]:
# Hierarchical categorical flattening
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

internet_features = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in internet_features:
    df[col] = df[col].replace("No internet service", "No")

for col in categorical_features:
    print(f"\n{col}")
    print("Counts:", df[col].value_counts().to_dict())


gender
Counts: {'Male': 3549, 'Female': 3483}

SeniorCitizen
Counts: {'No': 5890, 'Yes': 1142}

Partner
Counts: {'No': 3639, 'Yes': 3393}

Dependents
Counts: {'No': 4933, 'Yes': 2099}

PhoneService
Counts: {'Yes': 6352, 'No': 680}

MultipleLines
Counts: {'No': 4065, 'Yes': 2967}

InternetService
Counts: {'Fiber optic': 3096, 'DSL': 2416, 'No': 1520}

OnlineSecurity
Counts: {'No': 5017, 'Yes': 2015}

OnlineBackup
Counts: {'No': 4607, 'Yes': 2425}

DeviceProtection
Counts: {'No': 4614, 'Yes': 2418}

TechSupport
Counts: {'No': 4992, 'Yes': 2040}

StreamingTV
Counts: {'No': 4329, 'Yes': 2703}

StreamingMovies
Counts: {'No': 4301, 'Yes': 2731}

Contract
Counts: {'Month-to-month': 3875, 'Two year': 1685, 'One year': 1472}

PaperlessBilling
Counts: {'Yes': 4168, 'No': 2864}

PaymentMethod
Counts: {'Electronic check': 2365, 'Mailed check': 1604, 'Bank transfer (automatic)': 1542, 'Credit card (automatic)': 1521}


Numeric Sanity Check (TotalCharges vs Tenure)

In [12]:
calculated_total = df["MonthlyCharges"] * df["tenure"]
diff = df["TotalCharges"] - calculated_total

relative_error = diff.abs() / calculated_total.replace(0, np.nan)

# near matches within 5%
near_match = relative_error < 0.05

print(f"Near matches (<5% error): {near_match.sum()}")
print(f"Non-matches: {(~near_match).sum()}")

# flag strong anomalies (>20%)
anomalies = df[relative_error > 0.20]

print(f"\nStrong anomalies (>20% error): {len(anomalies)}")

print("\nRelative error statistics:")
print(relative_error.describe())

Near matches (<5% error): 5651
Non-matches: 1381

Strong anomalies (>20% error): 59

Relative error statistics:
count    7032.000000
mean        0.032017
std         0.039900
min         0.000000
25%         0.007220
50%         0.019997
75%         0.041734
max         0.573454
dtype: float64


In [13]:
df.to_csv(PROCESSED_DATA_PATH, index=False)